# 5CM522 Operational Research: start here

This site runs Jupyter and Python **inside your web browser**. Nothing is installed on your computer
and nothing is sent to a server: the code runs on your own machine.

**Run this notebook once at the start of each session** (menu *Run → Run All Cells*). If every
cell prints `OK`, all the course examples will run.

| What | Where |
| --- | --- |
| A week's material | `Lectures/Semester 1/Week 05 - …/` (each week has a `README.md`) |
| Slides | the week's `1 Lecture` folder |
| Notebooks to run in the lecture | the week's `2 Lecture Demos` folder, in number order |
| Tutorial sheet and answers | `3 Tutorial` and, after the practical, `4 Tutorial Answers` |
| Module handbook | `Handbook/` |

### Saving your work
Notebooks you edit here are stored **only in this browser**. Clearing browser data, using a
private window or switching computer loses them. Before you leave: right-click the notebook in the
file browser → **Download**.

## 1. Install the optimisation and simulation packages

In [ ]:
# Run this cell first (installs the extra packages; safe to re-run).
%pip install pyomo highspy simpy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

## 2. Check the versions

In [ ]:
import sys, numpy, scipy, pandas, matplotlib, pyomo, highspy, simpy
print("Python    ", sys.version.split()[0])
for mod in (numpy, scipy, pandas, matplotlib, pyomo, simpy):
    print(f"{mod.__name__:<10}", getattr(mod, "__version__", "") or mod.version.version)
print("OK")

## 3. Linear programme with Pyomo + HiGHS

maximise 3x + 2y subject to x + y ≤ 4, x + 3y ≤ 6, x, y ≥ 0. The answer is x = 4, y = 0, value 12.

The course scripts originally used the GLPK solver, which cannot run in a browser. Every notebook on
this site uses **HiGHS** instead: `SolverFactory('appsi_highs')`.

In [ ]:
import pyomo.environ as pyo

m = pyo.ConcreteModel()
m.x = pyo.Var(within=pyo.NonNegativeReals)
m.y = pyo.Var(within=pyo.NonNegativeReals)
m.obj = pyo.Objective(expr=3 * m.x + 2 * m.y, sense=pyo.maximize)
m.c1 = pyo.Constraint(expr=m.x + m.y <= 4)
m.c2 = pyo.Constraint(expr=m.x + 3 * m.y <= 6)

pyo.SolverFactory("appsi_highs").solve(m)
print("x =", pyo.value(m.x), " y =", pyo.value(m.y), " objective =", pyo.value(m.obj))
assert abs(pyo.value(m.obj) - 12) < 1e-6
print("OK")

## 4. A tiny SimPy simulation (used in Semester 2)

In [ ]:
import random
import simpy

random.seed(1)
waits = []

def customer(env, desk):
    arrive = env.now
    with desk.request() as req:
        yield req
        waits.append(env.now - arrive)
        yield env.timeout(random.expovariate(1 / 4))   # mean service 4 min

def arrivals(env, desk):
    while True:
        yield env.timeout(random.expovariate(1 / 5))   # mean gap 5 min
        env.process(customer(env, desk))

env = simpy.Environment()
desk = simpy.Resource(env, capacity=1)
env.process(arrivals(env, desk))
env.run(until=480)
print(f"{len(waits)} customers served, average wait {sum(waits) / len(waits):.1f} min")
print("OK")

## 5. A plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 6, 100)
plt.plot(x, 4 - x, label="x + y = 4")
plt.plot(x, (6 - x) / 3, label="x + 3y = 6")
plt.scatter([4], [0], color="black", zorder=3, label="optimum (4, 0)")
plt.xlim(0, 6); plt.ylim(0, 4.5); plt.xlabel("x"); plt.ylabel("y"); plt.legend()
plt.title("Feasible region corners")
plt.show()
print("OK")

## If something fails
- **`ModuleNotFoundError`**: run the install cell (step 1) again; installs last only for the open tab.
- **Kernel busy for a long time**: the first run downloads Python and the packages (tens of
  megabytes). Give it a minute; if nothing happens after that, reload the page.
- **Use Chrome, Edge or Firefox** (recent versions). Safari on iPad works but is slow.
- **Still stuck**: use Google Colab or Anaconda Jupyter; every notebook here also runs there.
- **The kernel dies ("table index is out of bounds")**: a notebook used SciPy's `linprog`/`milp` *and* Pyomo with HiGHS. Each brings its own copy of HiGHS and the browser cannot load both. Keep them in separate notebooks (the course examples already do).
